In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup
import pandas as pd
import time

In [7]:
driver = webdriver.Chrome()

In [8]:
url = "https://www.lamudi.co.id/properti/41032-73-2b88f097ff24-9218-197ee9b-9946-725c"

driver.get(url)

In [9]:
WebDriverWait(driver,30).until(
    EC.presence_of_element_located((By.TAG_NAME,"h1"))
)

<selenium.webdriver.remote.webelement.WebElement (session="e5671717aa00165599a2430f3d92c4c1", element="f.44C4F46367455AC3E183C59E79023AD3.d.501B21672358E89B8620EAD52BFBD74D.e.250")>

In [10]:
html = driver.page_source

soup = BeautifulSoup(html,"html.parser")

In [11]:
judul = soup.find("h1")

print(judul.get_text(strip=True))

Rumah disewakan cocok untuk keluarga


In [21]:
def ambil_text(soup, tag, class_name=None, data_test=None):
    attrs = {}
    if data_test:
        attrs["data-test"] = data_test

    elemen = soup.find(tag, class_=class_name, attrs=attrs)

    return elemen.get_text(strip=True) if elemen else None

In [40]:
kamar_tidur = ambil_text(
    soup,
    tag="div",
    class_name="details-item-value",
    data_test="bedrooms-value"
)

kamar_mandi = ambil_text(
    soup,
    tag="div",
    class_name="details-item-value",
    data_test="full-bathrooms-value"
)

luas_total = ambil_text(
    soup,
    tag="span",
    class_name="features-component__value",
    data_test="floor-area-value"
)

nama_agen = ambil_text(
    soup,
    tag="span",
    class_name="agency__name",
    data_test="agency-name"
)

In [36]:
print(kamar_tidur)

4 Kamar Tidur


In [37]:
print(kamar_mandi)

2 Kamar Mandi


In [41]:
print(luas_total)

150 m²


In [14]:
kategori = soup.find(attrs={"data-test":"property-type-value"})

print(kategori.get_text(strip=True))

Rumah


In [23]:
harga = ambil_text(
    soup,
    tag="div",
    class_name="prices-and-fees__price",
    data_test="listing-price"
)

print(harga)

Rp 2,10M


In [34]:
print(nama_agen)

REstate Gotong Royong 2


In [25]:
tanggal = soup.find("div", class_="date")

print(tanggal.get_text(" ", strip=True))

9 Jul 2025 - Diterbitkan oleh REstate Gotong Royong 2


In [26]:
tanggal = tanggal.get_text(" ", strip=True).split("-")[0].strip()

print(tanggal)

9 Jul 2025


In [28]:
lokasi = soup.find("div", class_="view-map__text")

lokasi = lokasi.get_text(strip=True) if lokasi else None

print(lokasi)

Kambang Barat, Pesisir Selatan, Sumatra Barat


In [29]:
bagian = [x.strip() for x in lokasi.split(",")]

kota = bagian[-2] if len(bagian) >= 2 else None
provinsi = bagian[-1] if len(bagian) >= 1 else None

print("Kota      :", kota)
print("Provinsi  :", provinsi)

Kota      : Pesisir Selatan
Provinsi  : Sumatra Barat


In [54]:
def scrape_detail(driver, url):

    driver.get(url)

    WebDriverWait(driver,30).until(
        EC.presence_of_element_located((By.TAG_NAME,"h1"))
    )

    soup = BeautifulSoup(driver.page_source,"html.parser")

    # Judul
    judul = ambil_text(
        soup,
        tag="h1"
    )

    # Harga
    harga = ambil_text(
        soup,
        tag="div",
        class_name="prices-and-fees__price",
        data_test="listing-price"
    )

    # Lokasi
    lokasi = soup.find("div", class_="view-map__text")
    lokasi = lokasi.get_text(strip=True) if lokasi else None

    # Kota & Provinsi
    bagian = [x.strip() for x in lokasi.split(",")] if lokasi else []

    kota = bagian[-2] if len(bagian) >= 2 else None
    provinsi = bagian[-1] if len(bagian) >= 1 else None

    # Tipe Properti
    tipe = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="property-type-value"
    )

    # Kamar Tidur
    kamar_tidur = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="bedrooms-value"
    )

    # Kamar Mandi
    kamar_mandi = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="full-bathrooms-value"
    )

    # Luas Total
    luas_total = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="floor-area-value"
    )

    # Luas Tanah
    luas_tanah = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="plot-area-value"
    )

    # Nama Agen
    nama_agen = ambil_text(
        soup,
        tag="span",
        class_name="agency__name",
        data_test="agency-name"
    )

    # Tanggal Posting
    tanggal = soup.find("div", class_="date")
    tanggal = tanggal.get_text(" ", strip=True) if tanggal else None

    if tanggal:
        tanggal = tanggal.split("-")[0].strip()

    return {
        "Judul": judul,
        "Harga": harga,
        "Lokasi": lokasi,
        "Kota": kota,
        "Provinsi": provinsi,
        "Tipe Properti": tipe,
        "Kamar Tidur": kamar_tidur,
        "Kamar Mandi": kamar_mandi,
        "Luas Total": luas_total,
        "Luas Tanah": luas_tanah,
        "Nama Agen": nama_agen,
        "Tanggal Posting": tanggal,
        "Link": url
    }

In [55]:
driver = webdriver.Chrome()

data = scrape_detail(
    driver,
    "https://www.lamudi.co.id/properti/41032-73-2b88f097ff24-9218-197ee9b-9946-725c"
)

driver.quit()

data

{'Judul': 'Rumah disewakan cocok untuk keluarga',
 'Harga': 'Rp 2,10M',
 'Lokasi': 'Kambang Barat, Pesisir Selatan, Sumatra Barat',
 'Kota': 'Pesisir Selatan',
 'Provinsi': 'Sumatra Barat',
 'Tipe Properti': 'Rumah',
 'Kamar Tidur': '4 Kamar Tidur',
 'Kamar Mandi': '2 Kamar Mandi',
 'Luas Total': '150 m²',
 'Luas Tanah': None,
 'Nama Agen': 'REstate Gotong Royong 2',
 'Tanggal Posting': '9 Jul 2025',
 'Link': 'https://www.lamudi.co.id/properti/41032-73-2b88f097ff24-9218-197ee9b-9946-725c'}

In [57]:
import pandas as pd

df = pd.read_csv("lamudi_page1.csv")

df.head()

,Judul,Harga,Lokasi,Link
0,Rumah Dijual di Pesisir Selatan,"Rp 2,10M","Kambang Barat, Pesisir Selatan, Sumatra Barat",https://www.lamudi.co.id/properti/41032-73-2b8...
1,Tanah Dijual di Pesisir Selatan,Rp 530Jt,"Kambang, Pesisir Selatan, Sumatra Barat",https://www.lamudi.co.id/properti/41032-73-38e...
2,Ruang Usaha Dijual di Depok,"Rp 5,50M","Manggung, Catur Tunggal, Depok, Sleman, Daerah...",https://www.lamudi.co.id/properti/41032-73-fab...
3,Ruang Usaha Dijual di Mlati,"Rp 5,70M","Sinduadi, Mlati, Sleman, Daerah Istimewa Yogya...",https://www.lamudi.co.id/properti/41032-73-dc6...
4,Rumah Dijual di Cikini,Rp 58M,"RW 02, Cikini, Menteng, Jakarta Pusat, Daerah ...",https://www.lamudi.co.id/properti/41032-73-ab3...


In [58]:
driver = webdriver.Chrome()

hasil = []

for url in df["Link"]:
    try:
        hasil.append(scrape_detail(driver, url))
    except Exception as e:
        print(f"Gagal: {url}")
        print(e)

driver.quit()

In [59]:
import pandas as pd

hasil_df = pd.DataFrame(hasil)

hasil_df.head()

,Judul,Harga,Lokasi,Kota,Provinsi,Tipe Properti,Kamar Tidur,Kamar Mandi,Luas Total,Luas Tanah,Nama Agen,Tanggal Posting,Link
0,Rumah disewakan cocok untuk keluarga,"Rp 2,10M","Kambang Barat, Pesisir Selatan, Sumatra Barat",Pesisir Selatan,Sumatra Barat,Rumah,4 Kamar Tidur,2 Kamar Mandi,150 m²,NaN,REstate Gotong Royong 2,9 Jul 2025,https://www.lamudi.co.id/properti/41032-73-2b8...
1,Lahan Pertanian 850 M2 di Kambang Barat,Rp 530Jt,"Kambang, Pesisir Selatan, Sumatra Barat",Pesisir Selatan,Sumatra Barat,Tanah,NaN,NaN,NaN,850 m²,REstate Gotong Royong 2,9 Jul 2025,https://www.lamudi.co.id/properti/41032-73-38e...
2,"Kost Dijual di Pogung Jogja, Kamar Mezanin - S...","Rp 5,50M","Manggung, Catur Tunggal, Depok, Sleman, Daerah...",Sleman,Daerah Istimewa Yogyakarta,Ruang Usaha,20 Kamar Tidur,21 Kamar Mandi,421 m²,193 m²,Harianto Property Jogja,8 Apr 2026,https://www.lamudi.co.id/properti/41032-73-fab...
3,Kost Esklusif 26 Kamar di Plemburan Dekat UGM ...,"Rp 5,70M","Sinduadi, Mlati, Sleman, Daerah Istimewa Yogya...",Sleman,Daerah Istimewa Yogyakarta,Ruang Usaha,26 Kamar Tidur,26 Kamar Mandi,NaN,252 m²,Harianto Property Jogja,8 Apr 2026,https://www.lamudi.co.id/properti/41032-73-dc6...
4,Dijual Rumah Mewah @Menteng Jakarta Pusat Luas...,Rp 58M,"RW 02, Cikini, Menteng, Jakarta Pusat, Daerah ...",Jakarta Pusat,Daerah Khusus Ibukota Jakarta,Rumah,5 Kamar Tidur,5 Kamar Mandi,550 m²,753 m²,Prestige SCBD Property,"2 minggu, 6 hari yang lalu",https://www.lamudi.co.id/properti/41032-73-ab3...


In [62]:
import os

os.makedirs("data", exist_ok=True)

In [2]:
hasil_df.to_csv(
    "data/lamudi_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Berhasil disimpan di data/lamudi_detail.csv")

NameError: name 'hasil_df' is not defined

In [3]:
print(hasil_df.shape)
hasil_df.head()

NameError: name 'hasil_df' is not defined